In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

The goal of this project was to study the network structure and evolution of citation papers. The research topic I used was AI but can be anything really

In [ ]:
import requests
import time
import json
import os
from tqdm import tqdm

import networkx as nx

In [ ]:
BASE_URL = "https://api.openalex.org/works"

CONCEPT_ID = "C154945302"  # AI
YEARS = list(range(2015, 2021))
PAPERS_PER_YEAR = 200

SAVE_PATH = "/kaggle/working/papers.json"

In [ ]:
#Query the openalex API to retrive AI research papers with their publication year and citations

def fetch_papers(year):
    params = {
        "filter": f"concepts.id:{CONCEPT_ID},publication_year:{year}",
        "sort": "cited_by_count:desc",
        "per-page": PAPERS_PER_YEAR
    }
    
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()["results"]


# Load existing data if it exists (resume capability)
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, "r") as f:
        papers = json.load(f)
    print(f"Loaded {len(papers)} cached papers")
else:
    papers = {}


for year in YEARS:
    # Skip if already fetched
    if any(p["year"] == year for p in papers.values()):
        print(f"Skipping {year} (already fetched)")
        continue

    print(f"Fetching {year}...")
    
    try:
        results = fetch_papers(year)
        
        for paper in results:
            papers[paper["id"]] = {
                "year": paper["publication_year"],
                "cited_by_count": paper["cited_by_count"],
                "references": paper["referenced_works"]
            }
        
        # Save after EACH year (critical for Kaggle)
        with open(SAVE_PATH, "w") as f:
            json.dump(papers, f)
        
        time.sleep(1)
    
    except Exception as e:
        print(f"Error on {year}: {e}")
        break

print(f"Total papers collected: {len(papers)}")

In [ ]:
with open(SAVE_PATH, "r") as f:
    papers = json.load(f)

print("Loaded papers:", len(papers))

In [ ]:
node_ids = set(papers.keys())
edges = []

for pid, data in papers.items():
    for ref in data["references"]:
        if ref in node_ids:
            edges.append((pid, ref))

print("Edges:", len(edges))

In [ ]:
G = nx.DiGraph()

G.add_nodes_from(node_ids)
G.add_edges_from(edges)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:
in_degrees = [d for _, d in G.in_degree()]

print("Average in-degree:", sum(in_degrees)/len(in_degrees))
print("Max in-degree:", max(in_degrees))

In [ ]:
with open("/kaggle/working/edges.json", "w") as f:
    json.dump(edges, f)

In [ ]:
import pandas as pd

df_nodes = pd.DataFrame.from_dict(papers, orient="index")
df_edges = pd.DataFrame(edges, columns=["source", "target"])

# Network Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#In degree is the amount of times a paper is cited by others in the network

in_degrees = np.array([d for _, d in G.in_degree()])

# Remove zeros for log plot
in_degrees = in_degrees[in_degrees > 0]

plt.figure()
plt.hist(in_degrees, bins=50)
plt.xlabel("In-degree")
plt.ylabel("Count")
plt.title("In-degree distribution")
plt.show()

In [ ]:
plt.figure()
plt.hist(in_degrees, bins=50, log=True)
plt.xscale("log")
plt.xlabel("In-degree (log)")
plt.ylabel("Frequency (log)")
plt.title("Log-Log Degree Distribution")
plt.show()

The network appears to be scale-free

In [ ]:
#Get the page rank for top papers (pargerank assigns higher values to influencial sources)

pagerank = nx.pagerank(G, alpha=0.85)

# Top 10 papers
top_papers = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]

for pid, score in top_papers:
    print(pid, score)

In [ ]:

#A look at some of the top papers
for pid, score in top_papers:
    print("Score:", score)
    print("Year:", papers[pid]["year"])
    print("Citations:", papers[pid]["cited_by_count"])
    print("-----")

In [ ]:
#We use the louvain algorithm to detect communities

!pip install python-louvain

In [ ]:
import community as community_louvain

# Convert to undirected (standard for Louvain)
G_undirected = G.to_undirected()

partition = community_louvain.best_partition(G_undirected)

# partition = {node_id: community_id}

In [ ]:
from collections import Counter

community_sizes = Counter(partition.values())

print("Number of communities:", len(community_sizes))
print("Largest communities:", community_sizes.most_common(10))

In [ ]:
from collections import defaultdict

community_papers = defaultdict(list)

for pid, comm_id in partition.items():
    community_papers[comm_id].append(pid)

In [ ]:
def top_in_community(comm_id, n=5):
    papers_in_comm = community_papers[comm_id]
    
    ranked = sorted(
        papers_in_comm,
        key=lambda pid: papers[pid]["cited_by_count"],
        reverse=True
    )
    
    return ranked[:n]


# Example
for pid in top_in_community(0):
    print(papers[pid]["year"], papers[pid]["cited_by_count"])

In [ ]:
community_years = defaultdict(list)

for pid, comm_id in partition.items():
    community_years[comm_id].append(papers[pid]["year"])

In [ ]:
# Get top 500 nodes by PageRank
top_nodes = sorted(pagerank, key=pagerank.get, reverse=True)[:500]

G_sub = G.subgraph(top_nodes)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 12))

pos = nx.spring_layout(G_sub, k=0.15, iterations=50)

In [ ]:
node_colors = [partition[n] for n in G_sub.nodes()]

In [ ]:
node_sizes = [pagerank[n] * 50000 for n in G_sub.nodes()]

In [ ]:
nx.draw_networkx_nodes(
    G_sub,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.tab20,
    alpha=0.8
)

nx.draw_networkx_edges(
    G_sub,
    pos,
    alpha=0.2,
    width=0.5
)

plt.title("Citation Network (Top Nodes by PageRank)")
plt.axis("off")
plt.show()

In [ ]:
pos = nx.spring_layout(G_sub, k=0.3)

In [ ]:
nx.write_gexf(G, "/kaggle/working/network.gexf")

In [ ]:
#Are core nodes more connected?
core_nodes = sorted(pagerank, key=pagerank.get, reverse=True)[:100]

avg_degree_core = sum(dict(G.degree(core_nodes)).values()) / len(core_nodes)
avg_degree_all = sum(dict(G.degree()).values()) / G.number_of_nodes()

print("Core avg degree:", avg_degree_core)
print("Overall avg degree:", avg_degree_all)

In [ ]:
from collections import Counter
print(Counter(partition.values()))

In [ ]:
betweenness = nx.betweenness_centrality(G_sub)

top_bridges = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]

for pid, score in top_bridges:
    print(pid, score)

In [ ]:
time_slices = [2016, 2017, 2018, 2019, 2020]

In [ ]:
def build_graph_up_to_year(year, papers):
    nodes = [pid for pid, data in papers.items() if data["year"] <= year]
    node_set = set(nodes)
    
    edges = []
    for pid in nodes:
        for ref in papers[pid]["references"]:
            if ref in node_set:
                edges.append((pid, ref))
    
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    G.add_edges_from(edges)
    
    return G

In [ ]:
results = []

for year in time_slices:
    G_t = build_graph_up_to_year(year, papers)
    
    num_nodes = G_t.number_of_nodes()
    num_edges = G_t.number_of_edges()
    
    avg_degree = sum(dict(G_t.degree()).values()) / num_nodes
    
    results.append({
        "year": year,
        "nodes": num_nodes,
        "edges": num_edges,
        "avg_degree": avg_degree
    })

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(results)

plt.figure()
plt.plot(df["year"], df["nodes"], label="Nodes")
plt.plot(df["year"], df["edges"], label="Edges")
plt.legend()
plt.title("Network Growth Over Time")
plt.show()

In [ ]:
import community as community_louvain

community_counts = []

for year in time_slices:
    G_t = build_graph_up_to_year(year, papers)
    
    G_u = G_t.to_undirected()
    partition_t = community_louvain.best_partition(G_u)
    
    num_communities = len(set(partition_t.values()))
    
    community_counts.append((year, num_communities))

In [ ]:
years, counts = zip(*community_counts)

plt.figure()
plt.plot(years, counts)
plt.title("Number of Communities Over Time")
plt.xlabel("Year")
plt.ylabel("Communities")
plt.show()

In [ ]:
years_to_plot = [2016, 2018, 2020]

In [ ]:
top_nodes_over_time = {}

for year in time_slices:
    G_t = build_graph_up_to_year(year, papers)
    pr = nx.pagerank(G_t)
    
    top_nodes = sorted(pr, key=pr.get, reverse=True)[:5]
    top_nodes_over_time[year] = top_nodes

In [ ]:
print(top_nodes_over_time)

In [ ]:
df["edge_node_ratio"] = df["edges"] / df["nodes"]

In [ ]:
plt.figure()
plt.plot(df["year"], df["edge_node_ratio"])
plt.title("Edges per Node Over Time")
plt.xlabel("Year")
plt.ylabel("Edges / Nodes")
plt.show()

#Network is densifying

In [ ]:
from collections import Counter

for year in time_slices:
    G_t = build_graph_up_to_year(year, papers)
    partition_t = community_louvain.best_partition(G_t.to_undirected())
    
    sizes = Counter(partition_t.values())
    print(year, sorted(sizes.values(), reverse=True)[:5])

In [ ]:
#Now we look at modularity,which measures how separated communities are

import community as community_louvain

modularity_results = []

for year in time_slices:
    G_t = build_graph_up_to_year(year, papers)
    
    # Convert to undirected for Louvain
    G_u = G_t.to_undirected()
    
    # Compute partition
    partition_t = community_louvain.best_partition(G_u)
    
    # Compute modularity
    mod_score = community_louvain.modularity(partition_t, G_u)
    
    modularity_results.append({
        "year": year,
        "modularity": mod_score
    })

In [ ]:
import pandas as pd

mod_df = pd.DataFrame(modularity_results)
print(mod_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(mod_df["year"], mod_df["modularity"], marker='o')
plt.title("Modularity Over Time")
plt.xlabel("Year")
plt.ylabel("Modularity")
plt.grid(True)
plt.show()

This is a very interesting structural change. Modularity decreased until 2017 then increased after. Later we will associate this with the widespread adoption of deep learning, often termed "the year of AI".

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(df["year"], df["edge_node_ratio"], label="Edge/Node Ratio")
ax1.set_ylabel("Edge/Node Ratio")

ax2 = ax1.twinx()
ax2.plot(mod_df["year"], mod_df["modularity"], color='orange', label="Modularity")
ax2.set_ylabel("Modularity")

plt.title("Network Densification vs Community Structure")
plt.show()

In [ ]:
def cross_community_ratio(G, partition):
    cross_edges = 0
    total_edges = G.number_of_edges()
    
    for u, v in G.edges():
        if partition[u] != partition[v]:
            cross_edges += 1
    
    return cross_edges / total_edges

In [ ]:
from collections import Counter

def avg_community_size(partition):
    sizes = Counter(partition.values())
    return sum(sizes.values()) / len(sizes)

In [ ]:
G_pre = build_graph_up_to_year(2016, papers)
G_post = build_graph_up_to_year(2020, papers)

In [ ]:
import community as community_louvain

part_pre = community_louvain.best_partition(G_pre.to_undirected())
part_post = community_louvain.best_partition(G_post.to_undirected())

In [ ]:
from collections import Counter

def summarize_partition(partition):
    sizes = Counter(partition.values())
    return {
        "num_communities": len(sizes),
        "largest": max(sizes.values()),
        "top5": sorted(sizes.values(), reverse=True)[:5]
    }

print("Pre-2017:", summarize_partition(part_pre))
print("Post-2017:", summarize_partition(part_post))

In [ ]:
def cross_community_ratio(G, partition):
    cross_edges = 0
    total_edges = G.number_of_edges()
    
    for u, v in G.edges():
        if partition[u] != partition[v]:
            cross_edges += 1
    
    return cross_edges / total_edges

print("Pre cross ratio:", cross_community_ratio(G_pre, part_pre))
print("Post cross ratio:", cross_community_ratio(G_post, part_post))

In [ ]:
def visualize_graph(G, partition, pagerank, title):
    top_nodes = sorted(pagerank, key=pagerank.get, reverse=True)[:400]
    G_sub = G.subgraph(top_nodes)
    
    pos = nx.spring_layout(G_sub, k=0.2)
    
    node_colors = [partition[n] for n in G_sub.nodes()]
    node_sizes = [pagerank[n] * 50000 for n in G_sub.nodes()]
    
    plt.figure(figsize=(8, 8))
    
    nx.draw_networkx_nodes(G_sub, pos,
                           node_size=node_sizes,
                           node_color=node_colors,
                           cmap=plt.cm.tab20,
                           alpha=0.8)
    
    nx.draw_networkx_edges(G_sub, pos, alpha=0.2)
    
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
pr_pre = nx.pagerank(G_pre)
pr_post = nx.pagerank(G_post)

visualize_graph(G_pre, part_pre, pr_pre, "Pre-2017 Network")
visualize_graph(G_post, part_post, pr_post, "Post-2017 Network")

In [ ]:
#We want to extract communities

from collections import defaultdict

def get_top_papers_by_community(partition, papers, n=5):
    comm_dict = defaultdict(list)
    
    for pid, comm in partition.items():
        comm_dict[comm].append(pid)
    
    top_per_comm = {}
    
    for comm, pids in comm_dict.items():
        ranked = sorted(
            pids,
            key=lambda pid: papers[pid]["cited_by_count"],
            reverse=True
        )
        top_per_comm[comm] = ranked[:n]
    
    return top_per_comm

In [ ]:
top_pre = get_top_papers_by_community(part_pre, papers)
top_post = get_top_papers_by_community(part_post, papers)

In [ ]:
print(top_pre)

In [ ]:
def largest_communities(partition, k=5):
    from collections import Counter
    sizes = Counter(partition.values())
    return sorted(sizes.values(), reverse=True)[:k]

print("Pre:", largest_communities(part_pre))
print("Post:", largest_communities(part_post))

In [ ]:
def top_k_fraction(partition, k=2):
    from collections import Counter
    sizes = Counter(partition.values())
    top_k = sorted(sizes.values(), reverse=True)[:k]
    return sum(top_k) / sum(sizes.values())

print("Pre top-2 fraction:", top_k_fraction(part_pre))
print("Post top-2 fraction:", top_k_fraction(part_post))

In [ ]:
from collections import Counter

sizes = Counter(part_post.values())

# Get the two largest community IDs
top_two_comms = [comm for comm, _ in sizes.most_common(2)]

print("Top communities:", top_two_comms)

In [ ]:
def get_community_papers(partition, target_comm):
    return [pid for pid, comm in partition.items() if comm == target_comm]

comm1_papers = get_community_papers(part_post, top_two_comms[0])
comm2_papers = get_community_papers(part_post, top_two_comms[1])

In [ ]:
def top_papers(paper_ids, papers, n=10):
    return sorted(
        paper_ids,
        key=lambda pid: papers[pid]["cited_by_count"],
        reverse=True
    )[:n]

top_comm1 = top_papers(comm1_papers, papers)
top_comm2 = top_papers(comm2_papers, papers)

In [ ]:
import requests

def fetch_title(paper_id):
    url = f"https://api.openalex.org/works/{paper_id.split('/')[-1]}"
    r = requests.get(url)
    return r.json().get("title", "N/A")

In [ ]:
def print_top_with_titles(top_list):
    for pid in top_list:
        title = fetch_title(pid)
        print(title)
        print("Citations:", papers[pid]["cited_by_count"])
        print("-----")

print("Community 1:")
print_top_with_titles(top_comm1)

print("\nCommunity 2:")
print_top_with_titles(top_comm2)

In [ ]:
def get_titles(paper_ids, max_n=100):
    titles = []
    for pid in paper_ids[:max_n]:
        titles.append(fetch_title(pid).lower())
    return titles

titles_comm1 = get_titles(comm1_papers, 100)
titles_comm2 = get_titles(comm2_papers, 100)

In [ ]:
from collections import Counter

keywords = [
    "reinforcement", "policy", "agent", "environment",
    "convolutional", "cnn", "image", "vision",
    "neural", "deep", "network",
    "learning", "training",
    "transformer", "attention"
]

In [ ]:
def keyword_freq(titles, keywords):
    counts = Counter()
    for title in titles:
        for kw in keywords:
            if kw in title:
                counts[kw] += 1
    return counts

freq1 = keyword_freq(titles_comm1, keywords)
freq2 = keyword_freq(titles_comm2, keywords)

print("Community 1:", freq1)
print("Community 2:", freq2)

In [ ]:
import re

def top_words(titles, n=20):
    words = []
    for title in titles:
        words += re.findall(r'\b[a-z]+\b', title)
    
    return Counter(words).most_common(n)

print("Top words community 1:", top_words(titles_comm1))
print("Top words community 2:", top_words(titles_comm2))

#In 2017, the AI research community underwent a major shift with the introduction of the transformer architecture ("Attention is all you need").“Post-2017, the AI citation network becomes dominated by two major communities. One corresponds to general deep learning methods, including reinforcement learning and classification tasks, while the other is strongly centered on computer vision, particularly image segmentation and convolutional architectures. This suggests that the field has reorganized into a methodological core alongside major application domains.”